In [20]:
KEEP_COLS = [
    # IDs / linkage
    "pat_id",
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "SOPInstanceUID",
    "tar_name",
    "save_name",       # or dcm_name, depending on what you use
    "dcm_name",
    "row_num",
    "WindowWidth",
    "WindowCenter",

    # geometry / position
    "InstanceNumber",
    "ImagePositionPatient",
    "ImageOrientationPatient",
    "SliceThickness",
    "PixelSpacing",
    "Rows",
    "Columns",
    "PatientPosition",

    # CT meta / covariates
    "body_region",
    "slice_status",
    "phase",
    "Manufacturer",
    "StudyDate",
    "StudyTime",
    "AcquisitionDate",
    "AcquisitionTime",
]


In [ ]:
from tqdm import tqdm
import os
import pandas as pd
from multiprocessing import Pool, cpu_count
import uuid
import csv   # ← needed for csv.QUOTE_ALL

# ====================================================================
# Paths
# ====================================================================
input_folder = "<PRIVATE_DATA_PATH>"
tmp_parquet_folder = "<PRIVATE_DATA_PATH>"
output_parquet = "<PRIVATE_DATA_PATH>"
output_csv = "<PRIVATE_DATA_PATH>"  # optional

os.makedirs(tmp_parquet_folder, exist_ok=True)

csv_files = [f for f in os.listdir(input_folder) if f.endswith(".csv")]
csv_files.sort()
print(f"Found {len(csv_files)} CSV files in {input_folder}")

missing_warning_printed = False


def process_one_csv(fname):
    try:
        fpath = os.path.join(input_folder, fname)
        df = pd.read_csv(
            fpath,
            dtype=str,
            keep_default_na=True,
            low_memory=False,
        )

        cols_available = [c for c in KEEP_COLS if c in df.columns]
        missing = sorted(set(KEEP_COLS) - set(cols_available))
        if missing and not missing_warning_printed:
            print(f"[WARN] Some requested columns not found in {fname}: {missing}")
            print("       Will keep intersection only.")
            # don't bother with global here; warning once is enough in practice

        if not cols_available:
            print(f"[WARN] No requested columns present in {fname}, skipping.")
            return None

        df = df[cols_available].copy()

        tmp_name = f"{os.path.splitext(fname)[0]}_{uuid.uuid4().hex}.parquet"
        tmp_path = os.path.join(tmp_parquet_folder, tmp_name)

        df.to_parquet(tmp_path, index=False)
        return tmp_path

    except Exception as e:
        print(f"[ERROR] {fname}: {e}")
        return None


if __name__ == "__main__":
    num_workers = min(8, cpu_count())
    print(f"Using {num_workers} workers")

    tmp_paths = []
    from multiprocessing import Pool

    with Pool(processes=num_workers) as pool:
        for tmp in tqdm(
            pool.imap_unordered(process_one_csv, csv_files, chunksize=1),
            total=len(csv_files),
            desc="Processing CSVs in parallel",
        ):
            if tmp is not None:
                tmp_paths.append(tmp)

    if not tmp_paths:
        print("No temp parquet files created. Nothing to merge.")
    else:
        dfs = []
        for p in tqdm(tmp_paths, desc="Reading temp parquets"):
            dfs.append(pd.read_parquet(p))

        merged = pd.concat(dfs, axis=0, ignore_index=True)
        print(f"Merged shape: {merged.shape}")

        merged.to_parquet(output_parquet, index=False)
        print(f"✅ Saved merged table (Parquet) → {output_parquet}")

        # Here is the CSV with full quoting
        merged.to_csv(output_csv, index=False, quoting=csv.QUOTE_ALL)
        print(f"✅ Saved merged table (CSV) → {output_csv}")




In [ ]:
df = pd.read_csv("<PRIVATE_DATA_PATH>")

In [ ]:
import pandas as pd
import csv

# ------------------------------------------------------------------
# 1. Load main merged file
# ------------------------------------------------------------------
df = pd.read_csv(
    "<PRIVATE_DATA_PATH>",
    dtype=str,
    keep_default_na=True,
    low_memory=False,
)

# ------------------------------------------------------------------
# 2. Load the large mapping file (with correct paths/names)
# ------------------------------------------------------------------
large_file_path = "<PRIVATE_DATA_PATH>"  # <- change to real path

map_df = pd.read_csv(
    large_file_path,
    dtype=str,
    keep_default_na=True,
    low_memory=False,
)

# Make sure these columns exist in map_df:
#   tar_name, dcm_name, save_name
# If the column with the correct name is not called "save_name",
# change it below.
map_df = map_df[["tar_name", "dcm_name", "save_name"]].copy()
map_df = map_df.rename(columns={"save_name": "save_name2"})

# ------------------------------------------------------------------
# 3. Merge: attach save_name2 by (tar_name, dcm_name)
# ------------------------------------------------------------------
df = df.merge(
    map_df,
    on=["tar_name", "dcm_name"],
    how="left",
)


# ------------------------------------------------------------------
# 5. Save updated merged file
# ------------------------------------------------------------------
out_csv = "<PRIVATE_DATA_PATH>/ct_relevant_slices_merged_with_save_name2.csv"

df.to_csv(out_csv, index=False, quoting=csv.QUOTE_ALL)
print(f"✅ Saved updated CSV → {out_csv}")


In [ ]:
import os
import pandas as pd

ROOTS = [
    "<PRIVATE_DATA_PATH>",
    "<PRIVATE_DATA_PATH>",
    "<PRIVATE_DATA_PATH>",
]
ROOTS2 = [
    "<PRIVATE_DATA_PATH>",
]

# -----------------------------
# Helper: generic path checker
# -----------------------------
def _check_file_in_roots(folder, filename, roots):
    """
    Given a folder + filename and a list of root dirs,
    return True if any root/folder/filename exists as a file.
    """
    for root in roots:
        file_path = os.path.join(root, folder, filename)
        if os.path.isfile(file_path):
            return True
    return False


# -----------------------------
# Fallback checker for save_name2
# -----------------------------
def check_save_name2(save_name2: str) -> bool:
    """
    Check save_name2 against ROOTS2 (and optionally ROOTS if you want).
    Assumes save_name2 is 'folder/filename'.
    """
    if not save_name2 or not isinstance(save_name2, str):
        return False

    if "/" not in save_name2:
        # If it's just a folder, you can decide what to do.
        # For now, treat as invalid for slice file.
        return False

    folder, filename = save_name2.split("/", 1)

    # First look in ROOTS2 (new location)
    if _check_file_in_roots(folder, filename, ROOTS2):
        return True

    # Optional: also check old ROOTS as fallback
    if _check_file_in_roots(folder, filename, ROOTS):
        return True

    return False


# -----------------------------
# Main checker for each row
# -----------------------------
def check_save_name(row):
    """
    Check whether slice file exists for this row.
    1) Try save_name in ROOTS
    2) If fail and save_name2 exists, try save_name2 in ROOTS2/ROOTS
    Returns True/False.
    """
    save_name = row.get("save_name", "")
    save_name2 = row.get("save_name2", "")

    if "/" in save_name:
        folder, filename = save_name.split("/", 1)

        # Try original ROOTS first
        if _check_file_in_roots(folder, filename, ROOTS):
            return True

        # If not found, try save_name2 if present
        if save_name2:
            return check_save_name2(save_name2)

        return False

    return False



In [ ]:
import os
import pandas as pd
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import csv

# ---------------------------------------------------------------
# Load merged CSV
# ---------------------------------------------------------------
df = pd.read_csv(
    "<PRIVATE_DATA_PATH>",
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)

# ---------------------------------------------------------------
# Wrapper for multiprocessing (cannot pass row objects directly)
# ---------------------------------------------------------------
def check_row(i):
    """Check a single row by index (needed for multiprocessing)."""
    row = df.iloc[i]
    return check_save_name(row)

# ---------------------------------------------------------------
# Parallel map
# ---------------------------------------------------------------
if __name__ == "__main__":
    n = len(df)
    workers = min(16, cpu_count())   # adjust depending on your compute node
    print(f"Running existence check on {n} slices using {workers} workers")

    with Pool(workers) as pool:
        results = list(
            tqdm(
                pool.imap(check_row, range(n), chunksize=1000),
                total=n,
                desc="Checking slice paths"
            )
        )

    # Add results to dataframe
    df["exists"] = results


    print("\n=================================================")
    print(f"Total slices:         {n}")
    print(f"Slices found:         {df['exists'].sum()}")
    print(f"Slices missing:       {(~df['exists']).sum()}")
    print(f"Saved updated file →  {out_csv}")
    print("=================================================")


In [ ]:
df